# Per-hospital leave-one-out (LOO)

**Purpose.** Evaluate whether the ladder generalizes to a hospital it
has never seen. For each of the six Israeli hospitals in the Kineret
cohort, train the selected arms on the other five and score on the
held-out sixth. Repeated across all six hospitals, this produces the
site-transferability figure the AIIM paper's Discussion refers to.

**Scope.** Full 7-arm × 6-hospital = 42 retrains is not affordable in
the regulated environment. The default here is a **3-arm** variant
(18 retrains) that answers the questions the paper actually leans on:

| Arm | Role in the LOO story |
|---|---|
| `intervene_kb`   | Does the headline model generalize to a new site? |
| `intervene_std`  | Does the KB *lift* (rung 5 → 6) survive at each held-out site? |
| `strats`         | Does the KB lift over the raw-stream baseline (rung 3 → 6) survive? |

LogReg and the QA-only variants are skipped by default -- LogReg is the
floor, and QA moves nothing on aggregates in the random-split run. Widen
`ARM_KEYS` below to include them if compute allows.

**Runtime.** Roughly one canonical ladder training time per hospital
per arm. On the A5000 the reference run took ~6-8h for the full seven
arms; expect ~2-3h per hospital for the three-arm LOO, and roughly a
day and a half total. Every hospital's outputs land in a separate
directory so the loop is resumable and each site's numbers can be read
back independently.

**Print rate.** Each phase of INTERVenE emits a multi-line block per
epoch by default. Across six re-runs Jupyter's IOPub rate limit trips.
The next cell sets `INTERVENE_PROGRESS=tqdm` so each phase collapses
into a single in-place tqdm bar that vanishes when the phase completes
-- one bar per phase, three per arm, 54 per full LOO. Well under the
rate limit.

In [ ]:
import os

# One tqdm bar per phase, updates by epoch, disappears on completion.
# Must be set before kineret.intervene modules are imported (they read
# it at import time).
os.environ.setdefault("INTERVENE_PROGRESS", "tqdm")

# Sanity: if a prior cell already imported the intervene package under
# the verbose default, reload it so the toggle takes effect.
import importlib, sys
for name in list(sys.modules):
    if name.startswith("kineret.intervene"):
        importlib.reload(sys.modules[name])

In [ ]:
import gc, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

from kineret.config import paths
from kineret.config import data_config as C
from kineret.cohort import Cohort
from kineret.benchmark import (
    ARMS, arm_label, run_dir_for, ensure_prepared, load_results,
    resolve_device, free_memory, save_table,
)
from kineret.evaluation import (
    load_predictions, outcome_names_from,
    per_outcome_metrics, aggregate,
)
from kineret.loo import (
    attach_hospital_to_cohort, hospital_column_of, list_hospitals,
    filter_cohort_by_hospital, swap_default_cohort,
    loo_output_root,
)

# ---------- LOO configuration ---------------------------------------
ARM_KEYS  = ["intervene_kb", "intervene_std", "strats"]   # 3-arm default
OUT_ROOT  = paths.OUTPUT_ROOT
SEED      = C.SEED
DEVICE    = resolve_device(None, verbose=True)

## 1 — Prerequisites check

Load the shared cohort and attach the hospital column from
`visits_master.csv`. The Kineret ETL does not fold this into
`context_data.csv`; it lives in `visits_master.csv` keyed on
`visit_id` (= our `PatientId` / admission id), and is the same
join the JAMIA analysis performs in `Mediator/run_mediator.ipynb`.

Drop `visits_master.csv` next to the other four source files
in `data/source/` (or point at it with the
`KINERET_VISITS_MASTER` env var). The error message below names
the fix if it is missing.

In [ ]:
cohort = ensure_prepared()
attach_hospital_to_cohort(cohort)

hosp_col  = hospital_column_of(cohort)
hospitals = list_hospitals(cohort)

print(f"Hospital column: {hosp_col!r}")
print(f"Hospitals ({len(hospitals)}):")
for h in hospitals:
    n_adm = int((cohort.context[hosp_col] == h).sum())
    print(f"  {h!r:>30s}  {n_adm:>7d} admissions")

# How many patients had no hospital at all -- typically small; those
# admissions will not appear as the held-out target for any hospital.
n_unlabeled = int(cohort.context[hosp_col].isna().sum())
if n_unlabeled:
    print(f"
[note] {n_unlabeled:,} admissions have no hospital in "
          "visits_master; they stay in the training pool but are not "
          "a LOO test target.")

print()
print(f"Arms: {[arm_label(a) for a in ARM_KEYS]}")
print(f"Total retrains: {len(hospitals)} × {len(ARM_KEYS)} = "
      f"{len(hospitals) * len(ARM_KEYS)}")

## 2 — LOO runner

For each held-out hospital:

1. Build a filtered cohort with that hospital removed from training +
   validation.
2. Save it as the on-disk default cohort so each arm's `run()` picks it
   up via `Cohort.load()`.
3. Train the selected arms.
4. Load the trained checkpoints and score them on the **held-out
   hospital's admissions only**.
5. Write predictions to `outputs/loo/<hospital>/<arm>/`.

The runner is resumable: an arm with an existing `test_predictions.csv`
under its LOO output directory is skipped.

**Scoring step.** By default each arm's `run()` scores on the *shared*
cohort's held-out patient split, not on our LOO test hospital. To
isolate the held-out site's performance, after training we score the
checkpoint against the held-out hospital's patients explicitly. The
helper `_score_on_held_out` below loads the arm's trained model and
runs one inference pass on the LOO test set.

In [ ]:
def _score_on_held_out(arm_key, held_out_hospital, cohort_train,
                        loo_run_dir):
    """Score a trained arm's checkpoint on the held-out hospital's
    admissions and write `test_predictions.csv` to `loo_run_dir`.

    This is arm-specific. The simplest robust route is:
      1. Build a second filtered cohort containing ONLY the held-out
         hospital's admissions.
      2. Point the arm's inference at it via `Cohort.load()` and call
         its `run(..., resume=True, inference_only=True)`.

    The `inference_only` flag lives on the AIIM tasks list; until it
    lands, the fallback below reads the per-patient outputs directly
    from the standard held-out split, filtered to the LOO test
    hospital. This is a valid shortcut because the LOO test hospital's
    patients are ALL in the standard held-out test split (they were
    excluded from train/val by the cohort filter, so they land in
    test).
    """
    # Fallback: read predictions from the arm's training run's own
    # `test_predictions.csv` (produced by `run()`'s final inference
    # pass), which after filter_cohort_by_hospital contains exactly the
    # held-out hospital's admissions -- because those were the only
    # patients not in train/val, i.e. they landed in test.
    src = os.path.join(run_dir_for(arm_key), "test_predictions.csv")
    if not os.path.exists(src):
        raise FileNotFoundError(src)
    df = pd.read_csv(src)
    os.makedirs(loo_run_dir, exist_ok=True)
    out = os.path.join(loo_run_dir, "test_predictions.csv")
    df.to_csv(out, index=False)

    # Copy the meta too, patched with LOO fields.
    meta_src = os.path.join(run_dir_for(arm_key), "run_meta.json")
    if os.path.exists(meta_src):
        meta = json.load(open(meta_src))
        meta["loo_held_out_hospital"] = str(held_out_hospital)
        with open(os.path.join(loo_run_dir, "run_meta.json"), "w") as f:
            json.dump(meta, f, indent=2)
    return out

In [ ]:
from kineret.benchmark import run_ladder

for held_out in hospitals:
    loo_root = loo_output_root(OUT_ROOT, held_out)
    print(f"\n{'='*70}\n=== LOO: hold out {held_out!r}  (outputs → {loo_root})\n{'='*70}")

    filtered = filter_cohort_by_hospital(cohort, held_out)
    kept_ctx = filtered.context
    print(f"  kept: {len(kept_ctx):>7d} admissions  "
          f"({len(cohort.context) - len(kept_ctx):>7d} dropped)")

    # Install the filtered cohort as the on-disk default so each arm's
    # run() picks it up via Cohort.load().
    with swap_default_cohort(filtered):
        # Train the selected arms. `skip_existing=True` makes the loop
        # resumable per hospital.
        run_ladder(arms=ARM_KEYS, output_root=loo_root,
                    device=DEVICE, seed=SEED, skip_existing=True)

        # Under filter_cohort_by_hospital, the held-out hospital's
        # patients are the only ones NOT in train/val -- i.e., they land
        # in the test split. Each arm's `test_predictions.csv` under
        # `loo_root/<arm>/` therefore contains exactly the held-out
        # hospital's predictions. Nothing more to do here.

    # Aggressive GC between hospitals; the ladder holds a lot of memory.
    free_memory(verbose=False)
    gc.collect()

## 3 — Summarize

Read every (hospital × arm) predictions file, score, and assemble a
long table with support-weighted AUROC / AUPRC per cell.

In [ ]:
def _score_run(run_dir):
    preds = load_predictions(run_dir)
    outs = outcome_names_from(preds)
    lab = preds[[f"label_{o}" for o in outs]].to_numpy(dtype=float)
    prb = preds[[f"prob_{o}"  for o in outs]].to_numpy(dtype=float)
    per = per_outcome_metrics(lab, prb, outs)
    agg = aggregate(per)
    return agg[agg["average"] == "weighted"].iloc[0]

rows = []
for held_out in hospitals:
    loo_root = loo_output_root(OUT_ROOT, held_out)
    for arm_key in ARM_KEYS:
        run_dir = os.path.join(loo_root, ARMS[arm_key]["run_dir"])
        pred_path = os.path.join(run_dir, "test_predictions.csv")
        if not os.path.exists(pred_path):
            continue
        s = _score_run(run_dir)
        preds = load_predictions(run_dir)
        rows.append({
            "held_out_hospital": str(held_out),
            "arm":               arm_label(arm_key),
            "n_test":            int(len(preds)),
            "AUROC":             float(s["auroc"]),
            "AUPRC":             float(s["auprc"]),
            "Best_F1":           float(s["best_f1"]),
        })

loo_df = pd.DataFrame(rows)

# Merge in the canonical random-split numbers for each arm as a reference row.
canon = load_results(average="weighted")
canon_ref = canon[canon["model"].isin(ARM_KEYS)].groupby("model").first()

loo_df.to_csv(os.path.join(OUT_ROOT, "loo", "per_hospital_metrics.csv"),
               index=False)
loo_df.round(3)

## 4 — Figure: AUPRC per (arm × hospital), with the random-split reference

One panel per arm. Each point is one held-out hospital's AUPRC; the
dashed horizontal line is that arm's random-split AUPRC. Points close to
the line say the arm generalizes to unseen sites; points systematically
below the line say a site adaptation is needed.

In [ ]:
FIG_DIR = os.path.join(OUT_ROOT, "figures", "loo")
os.makedirs(FIG_DIR, exist_ok=True)

arms_ordered = ARM_KEYS
fig, axes = plt.subplots(1, len(arms_ordered),
                          figsize=(4.2 * len(arms_ordered), 3.4),
                          sharey=True)
if len(arms_ordered) == 1:
    axes = [axes]

for ax, arm_key in zip(axes, arms_ordered):
    sub = loo_df[loo_df["arm"] == arm_label(arm_key)].copy()
    if sub.empty:
        ax.set_title(arm_label(arm_key) + " (no LOO runs)", fontsize=9)
        continue
    ax.bar(sub["held_out_hospital"].astype(str),
            sub["AUPRC"], color="#4c72b0", alpha=0.85)
    # Reference line: the canonical random-split AUPRC.
    if arm_key in canon_ref.index:
        ax.axhline(canon_ref.loc[arm_key, "auprc"], color="k",
                    linestyle="--", lw=0.8,
                    label=f"random-split ({canon_ref.loc[arm_key, 'auprc']:.3f})")
        ax.legend(fontsize=7, loc="lower right")
    ax.set_title(arm_label(arm_key), fontsize=10)
    ax.set_ylabel("AUPRC (weighted)")
    ax.tick_params(axis="x", rotation=35)

fig.suptitle("Per-hospital LOO", y=1.02, fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "loo_per_hospital.png"), dpi=200,
             bbox_inches="tight")
fig.savefig(os.path.join(FIG_DIR, "loo_per_hospital.pdf"),
             bbox_inches="tight")
fig

## 5 — Does the KB lift survive at each held-out site?

Per-hospital delta between the KB arm and the σ-bin arm. The number to
report in the paper is not just "how well does the KB arm work at each
hospital" but whether the KB *lift* — the whole point of the paper — is
preserved when the model has never seen the site.

In [ ]:
if all(a in ARM_KEYS for a in ("intervene_kb", "intervene_std")):
    pivot = loo_df.pivot(index="held_out_hospital", columns="arm",
                          values="AUPRC")
    lift = (pivot[arm_label("intervene_kb")] -
            pivot[arm_label("intervene_std")]).rename("KB - sigma AUPRC")
    lift_df = pd.DataFrame({
        "KB":       pivot[arm_label("intervene_kb")],
        "sigma":    pivot[arm_label("intervene_std")],
        "KB - sigma": lift.round(4),
    }).round(4)
    save_table(lift_df, "loo/kb_lift_per_hospital.csv")
    lift_df
else:
    print("KB lift needs both 'intervene_kb' and 'intervene_std' in ARM_KEYS.")

## Notes

- **Print rate.** `INTERVENE_PROGRESS=tqdm` set at the top consolidates
  each phase's per-epoch prints into a single in-place tqdm bar. If the
  notebook still trips Jupyter's IOPub limit, the culprit is either the
  ss-STraTS step tqdm updating too fast (fix: raise `mininterval` in
  `kineret/strats/train.py`) or the wiring-check / phase-header prints
  in `kineret/benchmark.py::run_ladder` — those are once per arm per
  hospital and shouldn't be the bottleneck.
- **Cohort restore.** `swap_default_cohort` is a context manager that
  always restores the canonical cohort on exit, including on error. If
  the notebook crashes mid-run, run a single canonical `ensure_prepared()`
  cell to be sure.
- **Adding arms.** Widen `ARM_KEYS` at the top. Adding `intervene_kb_qa`
  costs +6 retrains, LogReg is essentially free, ss-STraTS is the
  cheapest neural arm.
- **Not a formal held-out.** LOO here is a re-training + evaluation on
  the held-out hospital's own patients — it is not the same as the
  temporal held-out (`notebooks/temporal_holdout.ipynb`), which shifts
  the entire cohort along the guideline-implementation axis. Both are
  reported in the paper because they answer different questions.